# 16 ACT 端到端：训练、评估与失败阶段分析

本 Notebook 连接 ACT 的数据配置、保护续训、严格闭环评估和视频复核。课程权重在三个固定面板上取得 `15/30`，分组结果为 `3/10 + 4/10 + 8/10`；stable61 与 protected DAgger 分别作为 `7/30` 和 `2/30` 的对照。


In [1]:
from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess
import sys

try:
    from IPython.display import HTML, Markdown, display
except Exception:
    class Markdown(str):
        pass

    class HTML(str):
        pass

    def display(obj):
        print(obj)


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("请从 AMD ROCm 专题目录或 notebooks 子目录启动 Jupyter。")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/04mujoco复现ACT、Pi0、SmolVLA"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", "/path/to/model/checkpoints"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))

# The AMD teaching workflow should be runnable from local datasets/checkpoints.
# Avoid surprising network calls during class or when AUP/Radeon Cloud cannot
# reach Hugging Face.
os.environ.setdefault("HF_HUB_OFFLINE", os.environ.get("NOTEBOOK_HF_OFFLINE", "1"))
os.environ.setdefault("TRANSFORMERS_OFFLINE", os.environ.get("NOTEBOOK_HF_OFFLINE", "1"))
os.environ.setdefault("HF_DATASETS_OFFLINE", os.environ.get("NOTEBOOK_HF_OFFLINE", "1"))
os.environ.setdefault("HF_HOME", str(Path(os.environ.get("CACHE_ROOT", OUTPUT_ROOT / "cache")) / "huggingface"))
os.environ.setdefault("HF_DATASETS_CACHE", str(Path(os.environ["HF_HOME"]) / "datasets"))

def public_path(path):
    path = Path(path)
    replacements = [
        (TOPIC_ROOT, "$TOPIC_ROOT"),
        (PROJECT_ROOT, "$PROJECT_ROOT"),
        (DATA_ROOT, "$DATA_ROOT"),
        (MODEL_ROOT, "$MODEL_ROOT"),
        (OUTPUT_ROOT, "$OUTPUT_ROOT"),
    ]
    value = str(path)
    for root, label in sorted(replacements, key=lambda item: len(str(item[0])), reverse=True):
        root_value = str(root)
        if root_value and value.startswith(root_value):
            return label + value[len(root_value):]
    return value


print("TOPIC_ROOT = $TOPIC_ROOT")
print("PROJECT_ROOT =", public_path(PROJECT_ROOT))
print("DATA_ROOT =", public_path(DATA_ROOT))
print("MODEL_ROOT =", public_path(MODEL_ROOT))
print("OUTPUT_ROOT =", public_path(OUTPUT_ROOT))


TOPIC_ROOT = $TOPIC_ROOT
PROJECT_ROOT = $PROJECT_ROOT
DATA_ROOT = $DATA_ROOT
MODEL_ROOT = $MODEL_ROOT
OUTPUT_ROOT = $OUTPUT_ROOT


In [2]:
def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(public_path(x) if isinstance(x, (str, Path)) else str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


def show_json(path, max_chars=5000):
    path = Path(path)
    if not path.exists():
        print("文件不存在：", public_path(path))
        return None
    data = json.loads(path.read_text(encoding="utf-8"))
    text = json.dumps(data, ensure_ascii=False, indent=2)
    print(text[:max_chars] + ("\n..." if len(text) > max_chars else ""))
    return data


def show_video(filename, title):
    path = ASSET_DIR / filename
    display(Markdown(f"**{title}**"))
    if path.exists():
        rel = f"../assets/{filename}"
        display(HTML(f"<video controls muted preload='metadata' width='960'><source src='{rel}' type='video/mp4'></video>"))
    else:
        print("缺少视频素材：", public_path(path))


def show_image(filename, title, width=960):
    path = ASSET_DIR / filename
    display(Markdown(f"**{title}**"))
    if path.exists():
        rel = f"../assets/{filename}"
        display(HTML(f"<img src='{rel}' width='{width}'>"))
    else:
        print("缺少图片素材：", public_path(path))


def run_cmd_preview(command, cwd=None):
    shown = [public_path(x) if isinstance(x, (str, Path)) else x for x in command]
    print("$", shlex.join([str(x) for x in shown]))
    if cwd:
        print("cwd =", public_path(cwd))


def tail_log(log_path, lines=40):
    path = Path(log_path)
    if not path.exists():
        print("日志不存在：", public_path(path))
        return
    content = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(content[-lines:]))


def env_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


RUN_SMOKE = env_flag("RUN_SMOKE")
RUN_LONG_TRAIN = env_flag("RUN_LONG_TRAIN")
RUN_EVAL = env_flag("RUN_EVAL")
EVAL_SCRIPT = Path(os.environ.get("EVAL_SCRIPT", PROJECT_ROOT / "eval_policy_success.py"))


_XVFB_PROCESS = None


def ensure_xvfb_display():
    """Start a lightweight virtual display for headless MuJoCo evaluation."""
    global _XVFB_PROCESS
    if os.environ.get("DISPLAY"):
        print("DISPLAY =", os.environ["DISPLAY"])
        return None
    xvfb_bin = shutil.which("Xvfb")
    if not xvfb_bin:
        print("没有发现 Xvfb；如遇 GLFW DISPLAY 报错，请先安装 xvfb。")
        return None
    display_id = os.environ.get("NOTEBOOK_XVFB_DISPLAY", ":99")
    _XVFB_PROCESS = subprocess.Popen(
        [xvfb_bin, display_id, "-screen", "0", "1280x1024x24"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    os.environ["DISPLAY"] = display_id
    print("已启动 Notebook 内部 Xvfb：DISPLAY =", display_id)
    return _XVFB_PROCESS


def ensure_project_layout():
    required = [PROJECT_ROOT / "asset" / "example_scene_y2.xml", PROJECT_ROOT / "mujoco_env"]
    missing = [path for path in required if not path.exists()]
    if missing:
        print("当前 PROJECT_ROOT 还不是可运行工程，缺少：")
        for path in missing:
            print(" -", public_path(path))
        print("请先设置 PROJECT_ROOT，再运行训练或评估单元。")
        return False
    return True


def write_json_yaml(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        import yaml
        text = yaml.safe_dump(payload, allow_unicode=True, sort_keys=False)
    except Exception:
        text = json.dumps(payload, ensure_ascii=False, indent=2) + "\n"
    path.write_text(text, encoding="utf-8")
    print("写出配置：", public_path(path))
    return path


def make_lerobot_train_config(policy_type, dataset_repo_id, dataset_root, output_dir, steps, batch_size, chunk_size, n_action_steps, seed=42):
    save_freq = int(os.environ.get(f"{policy_type.upper()}_SAVE_FREQ", os.environ.get("SAVE_FREQ", str(steps))))
    return {
        "dataset": {
            "repo_id": dataset_repo_id,
            "root": str(dataset_root),
            "use_imagenet_stats": True,
        },
        "policy": {
            "type": policy_type,
            "chunk_size": int(chunk_size),
            "n_action_steps": int(n_action_steps),
            "device": "cuda",
        },
        "output_dir": str(output_dir),
        "job_name": Path(output_dir).name,
        "batch_size": int(batch_size),
        "steps": int(steps),
        "save_freq": max(1, save_freq),
        "log_freq": 20,
        "num_workers": 4,
        "seed": int(seed),
        "resume": False,
        "eval_freq": -1,
        "save_checkpoint": True,
        "use_policy_training_preset": True,
        "wandb": {"enable": False, "disable_artifact": True},
    }


def train_lerobot_config_in_notebook(config_path, enabled=False, progress_name="train"):
    """Run LeRobot offline training directly inside the notebook kernel.

    The notebook cell owns dataset creation, policy creation, optimizer steps,
    checkpoint saving, tqdm progress, and metric JSONL writing.
    """
    config_path = Path(config_path)
    print("config =", public_path(config_path))
    if not enabled:
        print("未启动。设置 RUN_SMOKE=1 或 RUN_LONG_TRAIN=1 后，本单元会直接在 Notebook 内训练。")
        return None
    if not ensure_project_layout():
        return None

    import time
    from contextlib import nullcontext

    import draccus
    import torch
    from torch.amp import GradScaler
    from tqdm.auto import tqdm

    from lerobot.common.datasets.factory import make_dataset
    from lerobot.common.datasets.sampler import EpisodeAwareSampler
    from lerobot.common.optim.factory import make_optimizer_and_scheduler
    from lerobot.common.policies.factory import make_policy
    from lerobot.common.policies.utils import get_device_from_parameters
    from lerobot.common.utils.random_utils import set_seed
    from lerobot.common.utils.train_utils import get_step_checkpoint_dir, save_checkpoint, update_last_checkpoint
    from lerobot.common.utils.utils import get_safe_torch_device
    from lerobot.configs.train import TrainPipelineConfig

    cfg = draccus.parse(TrainPipelineConfig, config_path=config_path, args=[])
    cfg.validate()
    if cfg.seed is not None:
        set_seed(cfg.seed)

    device = get_safe_torch_device(cfg.policy.device, log=True)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

    print("Creating dataset...")
    dataset = make_dataset(cfg)
    print("Creating policy...")
    pretrained_override = os.environ.get(f"{cfg.policy.type.upper()}_PRETRAINED_PATH_OVERRIDE") or os.environ.get("POLICY_PRETRAINED_PATH_OVERRIDE")
    if pretrained_override and not cfg.resume:
        cfg.policy.pretrained_path = str(Path(pretrained_override))
        print("pretrained override =", public_path(cfg.policy.pretrained_path))
    elif cfg.policy.type == "pi0" and not cfg.resume:
        cfg.policy.pretrained_path = "lerobot/pi0"
    elif cfg.policy.type == "smolvla" and not cfg.resume:
        smolvla_base_candidates = [
            os.environ.get("SMOLVLA_BASE_PATH"),
            os.environ.get("SMOLVLA_PRETRAINED_BASE_PATH"),
            str(MODEL_ROOT / "smolvla_base" / "pretrained_model"),
            str(MODEL_ROOT / "lerobot_smolvla_base_legacy"),
            str(MODEL_ROOT / "lerobot_smolvla_base"),
        ]
        local_smolvla_base = next((Path(p) for p in smolvla_base_candidates if p and Path(p).exists()), None)
        if local_smolvla_base is not None:
            cfg.policy.pretrained_path = str(local_smolvla_base)
            print("local smolvla base =", public_path(cfg.policy.pretrained_path))
        else:
            cfg.policy.pretrained_path = "lerobot/smolvla_base"
    policy = make_policy(cfg=cfg.policy, ds_meta=dataset.meta)

    # Compatibility for newer Transformers: PaliGemmaForConditionalGeneration may expose
    # language_model as GemmaModel directly, while this LeRobot Pi0 code expects
    # language_model.model.  Use a non-Module proxy so checkpoints/state_dict stay clean.
    if cfg.policy.type == "pi0":
        try:
            lm = policy.model.paligemma_with_expert.paligemma.language_model
            if not hasattr(lm, "model"):
                class _LanguageModelCoreProxy:
                    def __init__(self, core):
                        self._core = core

                    def __getattr__(self, name):
                        return getattr(self._core, name)

                object.__setattr__(lm, "model", _LanguageModelCoreProxy(lm))
                print("patched Pi0 PaliGemma language_model.model compatibility proxy")
        except Exception as exc:
            print(f"Pi0 PaliGemma compatibility patch skipped: {exc}")

    policy.to(device)
    policy.train()

    optimizer, lr_scheduler = make_optimizer_and_scheduler(cfg, policy)
    grad_scaler = GradScaler(device.type, enabled=cfg.policy.use_amp)

    def _dataset_column_values(name):
        hf_dataset = getattr(dataset, "hf_dataset", None)
        if hf_dataset is None or name not in getattr(hf_dataset, "column_names", []):
            return None
        values = hf_dataset[name]
        try:
            return list(values)
        except TypeError:
            return [values[i] for i in range(len(values))]

    def _task_name_map():
        meta = getattr(dataset, "meta", None)
        tasks = getattr(meta, "tasks", None)
        if tasks is None:
            return {}
        if isinstance(tasks, dict):
            return {int(k): str(v) for k, v in tasks.items()}
        try:
            return {int(k): str(v) for k, v in dict(tasks).items()}
        except Exception:
            return {}

    def _make_weighted_sampler(generator):
        mode = os.environ.get("NOTEBOOK_FRAME_WEIGHT_MODE", "").strip().lower()
        if not mode or mode in {"0", "none", "off", "false"}:
            return None, {"mode": "none"}
        weights = torch.ones(len(dataset), dtype=torch.double)
        info = {"mode": mode, "num_frames": len(dataset)}

        if "blue" in mode:
            blue_weight = float(os.environ.get("NOTEBOOK_BLUE_WEIGHT", "2.0"))
            mask = [False] * len(dataset)
            task_indices = _dataset_column_values("task_index")
            task_names = _task_name_map()
            if task_indices is not None and task_names:
                for idx, task_index in enumerate(task_indices):
                    task_text = task_names.get(int(task_index), "").lower()
                    mask[idx] = ("blue" in task_text) or ("蓝" in task_text)
            else:
                for column in ["task", "language_instruction", "instruction"]:
                    values = _dataset_column_values(column)
                    if values is None:
                        continue
                    for idx, value in enumerate(values):
                        text = str(value).lower()
                        mask[idx] = ("blue" in text) or ("蓝" in text)
                    break
            blue_count = int(sum(mask))
            if blue_count == 0:
                print("警告：NOTEBOOK_FRAME_WEIGHT_MODE=blue 但没有识别到 blue/蓝 指令帧，采样退回均匀。")
            else:
                for idx, is_blue in enumerate(mask):
                    if is_blue:
                        weights[idx] *= blue_weight
            info.update({"blue_weight": blue_weight, "blue_frames": blue_count})

        weight_file = os.environ.get("NOTEBOOK_FRAME_WEIGHT_JSON")
        if weight_file:
            payload = json.loads(Path(weight_file).read_text(encoding="utf-8"))
            for key, value in payload.items():
                weights[int(key)] *= float(value)
            info.update({"weight_json": public_path(weight_file), "json_entries": len(payload)})

        if float(weights.sum()) <= 0:
            raise ValueError("采样权重总和为 0。")
        sampler = torch.utils.data.WeightedRandomSampler(
            weights=weights,
            num_samples=len(weights),
            replacement=True,
            generator=generator,
        )
        info.update(
            {
                "weight_min": float(weights.min()),
                "weight_max": float(weights.max()),
                "weight_mean": float(weights.mean()),
            }
        )
        return sampler, info

    generator = torch.Generator()
    if cfg.seed is not None:
        generator.manual_seed(int(cfg.seed))

    weighted_sampler, sampler_info = _make_weighted_sampler(generator)
    if weighted_sampler is not None:
        shuffle = False
        sampler = weighted_sampler
        print("Notebook weighted sampler =", json.dumps(sampler_info, ensure_ascii=False))
    elif hasattr(cfg.policy, "drop_n_last_frames"):
        shuffle = False
        sampler = EpisodeAwareSampler(
            dataset.episode_data_index,
            drop_n_last_frames=cfg.policy.drop_n_last_frames,
            shuffle=True,
        )
    else:
        shuffle = True
        sampler = None

    dataloader = torch.utils.data.DataLoader(
        dataset,
        num_workers=cfg.num_workers,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        sampler=sampler,
        generator=generator if sampler is None else None,
        pin_memory=device.type != "cpu",
        drop_last=False,
    )
    dl_iter = iter(dataloader)

    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = output_dir / "notebook_train_metrics.jsonl"
    num_learnable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    num_total = sum(p.numel() for p in policy.parameters())
    print(f"output_dir = {public_path(output_dir)}")
    print(f"steps = {cfg.steps}, batch_size = {cfg.batch_size}, frames = {dataset.num_frames}, episodes = {dataset.num_episodes}")
    print(f"learnable_params = {num_learnable:,}, total_params = {num_total:,}")

    last_metrics = None
    progress = tqdm(range(1, cfg.steps + 1), desc=progress_name, dynamic_ncols=True)
    start_all = time.perf_counter()
    for step in progress:
        load_start = time.perf_counter()
        try:
            batch = next(dl_iter)
        except StopIteration:
            dl_iter = iter(dataloader)
            batch = next(dl_iter)
        data_s = time.perf_counter() - load_start
        for key, value in batch.items():
            if isinstance(value, torch.Tensor):
                batch[key] = value.to(device, non_blocking=True)

        update_start = time.perf_counter()
        device_from_policy = get_device_from_parameters(policy)
        with torch.autocast(device_type=device_from_policy.type) if cfg.policy.use_amp else nullcontext():
            loss, output_dict = policy.forward(batch)
        grad_scaler.scale(loss).backward()
        grad_scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(
            policy.parameters(),
            cfg.optimizer.grad_clip_norm,
            error_if_nonfinite=False,
        )
        grad_scaler.step(optimizer)
        grad_scaler.update()
        optimizer.zero_grad()
        if lr_scheduler is not None:
            lr_scheduler.step()
        if hasattr(policy, "update"):
            policy.update()
        update_s = time.perf_counter() - update_start

        is_log_step = cfg.log_freq > 0 and (step % cfg.log_freq == 0 or step == 1 or step == cfg.steps)
        is_saving_step = cfg.save_checkpoint and (step % cfg.save_freq == 0 or step == cfg.steps)
        if is_log_step:
            last_metrics = {
                "step": step,
                "loss": float(loss.detach().cpu()),
                "grad_norm": float(grad_norm.detach().cpu()) if hasattr(grad_norm, "detach") else float(grad_norm),
                "lr": float(optimizer.param_groups[0]["lr"]),
                "update_s": float(update_s),
                "data_s": float(data_s),
                "elapsed_s": float(time.perf_counter() - start_all),
            }
            with metrics_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(last_metrics, ensure_ascii=False) + "\n")
            progress.set_postfix(
                loss=f"{last_metrics['loss']:.4f}",
                lr=f"{last_metrics['lr']:.1e}",
                updt_s=f"{last_metrics['update_s']:.3f}",
            )
        if is_saving_step:
            checkpoint_dir = get_step_checkpoint_dir(cfg.output_dir, cfg.steps, step)
            print(f"\nSaving checkpoint at step {step}: {public_path(checkpoint_dir)}")
            save_checkpoint(checkpoint_dir, step, cfg, policy, optimizer, lr_scheduler)
            update_last_checkpoint(checkpoint_dir)

    print("训练完成。metrics =", public_path(metrics_path))
    if last_metrics is not None:
        print(json.dumps(last_metrics, ensure_ascii=False, indent=2))
    return {"output_dir": output_dir, "metrics_path": metrics_path, "last_metrics": last_metrics}


def load_eval_module():
    import importlib.util

    if not EVAL_SCRIPT.exists():
        raise FileNotFoundError(f"评估脚本不存在：{public_path(EVAL_SCRIPT)}")
    spec = importlib.util.spec_from_file_location("notebook_eval_policy_success", EVAL_SCRIPT)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def run_eval_policy_in_notebook(
    policy_name,
    policy_path,
    result_path,
    episodes,
    seed_start,
    render=False,
    enabled=False,
    repo_id=None,
    dataset_root=None,
):
    print("policy =", policy_name)
    print("policy_path =", public_path(policy_path))
    print("result =", public_path(result_path))
    if not enabled:
        print("未启动。设置 RUN_EVAL=1 后，本单元会在 Notebook 内直接加载策略并闭环评估。")
        return None
    if not ensure_project_layout():
        return None

    import argparse
    from contextlib import contextmanager
    from tqdm.auto import tqdm

    @contextmanager
    def pushd(path):
        old = Path.cwd()
        os.chdir(path)
        try:
            yield
        finally:
            os.chdir(old)

    ensure_xvfb_display()
    module = load_eval_module()
    result_path = Path(result_path)
    result_path.parent.mkdir(parents=True, exist_ok=True)
    if result_path.exists():
        result_path.unlink()

    args = argparse.Namespace(
        policy=policy_name,
        episodes=int(episodes),
        seed_start=int(seed_start),
        max_action_steps=int(os.environ.get("ACT_EVAL_MAX_ACTION_STEPS", os.environ.get("EVAL_MAX_ACTION_STEPS", "400"))),
        hz=float(os.environ.get("EVAL_HZ", "20")),
        render=bool(render),
        output_jsonl=result_path,
        device=os.environ.get("EVAL_DEVICE", "cuda"),
        reset_policy_each_action=env_flag("EVAL_RESET_POLICY_EACH_ACTION", False),
        act_n_action_steps=(int(os.environ["ACT_EVAL_N_ACTION_STEPS"]) if os.environ.get("ACT_EVAL_N_ACTION_STEPS") else None),
        act_force_dataset_gripper=env_flag("ACT_EVAL_FORCE_DATASET_GRIPPER", False),
        act_clamp_timestamp=env_flag("ACT_EVAL_CLAMP_TIMESTAMP", False),
        act_policy_path=Path(policy_path),
        act_repo_id=repo_id or "datawhale_eai_pnp",
        act_dataset_root=Path(dataset_root or "./demo_data"),
        act_episode_timestamp_offsets=os.environ.get("ACT_EVAL_EPISODE_TIMESTAMP_OFFSETS", ""),
        act_episode_source_flags=os.environ.get("ACT_EVAL_EPISODE_SOURCE_FLAGS", ""),
        physical_success=env_flag("ACT_EVAL_PHYSICAL_SUCCESS", env_flag("EVAL_PHYSICAL_SUCCESS", True)),
        physical_min_lift=float(os.environ.get("EVAL_PHYSICAL_MIN_LIFT", "0.06")),
        physical_min_lift_steps=int(os.environ.get("EVAL_PHYSICAL_MIN_LIFT_STEPS", "3")),
        physical_final_upright_cos=float(os.environ.get("EVAL_PHYSICAL_FINAL_UPRIGHT_COS", "0.85")),
        smolvla_policy_path=Path(policy_path),
        pi0_policy_path=Path(policy_path),
        pi0_repo_id=repo_id or os.environ.get("PI0_DATASET_REPO_ID", "datawhale_eai_pnp_language"),
        pi0_dataset_root=Path(dataset_root or os.environ.get("PI0_DATASET_ROOT", "./demo_data_language")),
    )

    with pushd(PROJECT_ROOT):
        if policy_name == "act":
            policy = module.make_act_policy(
                args.device,
                args.act_policy_path,
                args.act_repo_id,
                args.act_dataset_root,
                n_action_steps=args.act_n_action_steps,
                episode_timestamp_offsets=args.act_episode_timestamp_offsets,
                episode_source_flags=args.act_episode_source_flags,
            )
            rollout = module.rollout_act
        elif policy_name == "smolvla":
            policy = module.make_smolvla_policy(args.device, args.smolvla_policy_path)
            rollout = module.rollout_language_policy
        elif policy_name == "pi0":
            policy = module.make_pi0_policy(args.device, args.pi0_policy_path, args.pi0_repo_id, args.pi0_dataset_root)
            rollout = module.rollout_language_policy
        else:
            raise ValueError(policy_name)

        rows = []
        for offset in tqdm(range(args.episodes), desc=f"{policy_name} eval", dynamic_ncols=True):
            seed = args.seed_start + offset
            row = rollout(args, policy, seed)
            rows.append(row)
            with result_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
            print(json.dumps(row, ensure_ascii=False))
    summarize_jsonl(result_path)
    return rows


def list_checkpoints(run_dir):
    run_dir = Path(run_dir)
    candidates = []
    for pattern in ["checkpoints/*/pretrained_model", "checkpoint*/pretrained_model", "*/pretrained_model", "pretrained_model"]:
        candidates.extend(run_dir.glob(pattern))
    unique = sorted(set(candidates))
    if not unique:
        print("尚未发现 checkpoint：", public_path(run_dir))
        return []
    for path in unique:
        print(" -", public_path(path))
    return unique


def resolve_eval_policy(default_path, trained_run_dir=None, env_name=None):
    if env_name and os.environ.get(env_name):
        path = Path(os.environ[env_name])
        print("评估使用环境变量指定权重：", public_path(path))
        return path
    if trained_run_dir is not None and env_flag("EVAL_USE_LONG_TRAIN"):
        checkpoints = list_checkpoints(trained_run_dir)
        if checkpoints:
            path = checkpoints[-1]
            print("评估使用本次长训最新 checkpoint：", public_path(path))
            return path
        print("未找到本次长训 checkpoint，回退到保护权重。")
    path = Path(default_path)
    print("评估使用保护/预训练权重：", public_path(path))
    return path


def summarize_jsonl(path):
    path = Path(path)
    if not path.exists():
        print("结果 JSONL 尚不存在：", public_path(path))
        return None
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    total = len(rows)
    legacy = sum(bool(row.get("success") or row.get("legacy_success")) for row in rows)
    if rows and all("physical_success" in row for row in rows):
        physical_count = sum(bool(row.get("physical_success")) for row in rows)
        physical_text = str(physical_count) + "/" + str(total)
    else:
        physical_text = "未记录"
    md_table(
        ["结果文件", "episodes", "legacy_success", "physical_success"],
        [(public_path(path), total, f"{legacy}/{total}", physical_text)],
    )
    return rows


## Checkpoint 1：课程结果

In [ ]:
rows = [
    ("repair15 保护续训", "15/30", "AMD395；step1500；固定面板 3/10 + 4/10 + 8/10"),
    ("stable61", "7/30", "AMD395；step2500；基础对照"),
    ("protected DAgger", "2/30", "相同严格评估协议下的数据配方对照"),
]
md_table(["训练配方", "严格成功率", "说明"], rows)


## Checkpoint 2：生成配置并启动训练

ACT smoke 通常需要 1–3 分钟；`ACT_STEPS=5000` 的运行时间由批量大小、图像编码器和设备决定。训练完成后，严格评估单元直接读取当前输出目录中的 checkpoint。


## Checkpoint 2.1：保护训练配方

`ACT_RECIPE=repair15` 选择低学习率续训、chunk20/n10、no-VAE、timestamp/object-init state 和纠偏数据降权。普通训练与保护训练使用独立输出目录。


In [ ]:
# PROTECTED_RECIPE_CELL
rows = [
    ("教学默认", "demo_data_language", "chunk50/n50/VAE", "ACT_STEPS=5000", "用于观察基础训练与闭环行为"),
    ("repair15", "act_base72_plus_dagger3x3_rebuild_v1", "chunk20/n10/no-VAE；timestamp+obj_init；纠偏降权", "2500 continuation steps；15/30"),
    ("stable61", "act_base49_plus_prefix23_rebuild_v1", "offset2/weight025", "5000 steps；7/30"),
    ("protected DAgger", "act_base72_plus_dagger3x3_rebuild_v1", "best025 配方", "5000 steps；2/30"),
]md_table(["模式", "数据", "训练策略", "训练与评估"], rows)

act_recipe = os.environ.get("ACT_RECIPE", "stable61").strip().lower()
is_repair = act_recipe in {"repair15", "stable61_to_dagger", "nomemleak"}
is_dagger = act_recipe in {"dagger", "dagger_best025", "protected"} and not is_repair
recipe_data = "act_base72_plus_dagger3x3_rebuild_v1" if is_repair else ("act_dagger_meta" if is_dagger else "demo_data_language")
recipe_model = "act_stable61_to_dagger_nomemleak_step1500_strict15of30" if is_repair else ("act_dagger_best025_rebuild_s5000" if is_dagger else "act_stable61_step2500")
protected_env = {
    "TEACHING_RECIPE": "protected-repair15" if is_repair else ("protected-dagger" if is_dagger else "protected-fallback"),
    "ACT_RECIPE": act_recipe,
    "ACT_TRAIN_DATA_ROOT": str(DATA_ROOT / recipe_data),
    "ACT_STEPS": "2500 continuation" if is_repair else "5000",
    "ACT_BATCH_SIZE": "16" if is_repair else "8",
    "ACT_EVAL_EPISODES": "30",
    "ACT_EVAL_SEED_START": "1030",
    "ACT_EVAL_N_ACTION_STEPS": "10" if (is_dagger or is_repair) else "50",
    "ACT_EVAL_CLAMP_TIMESTAMP": "1" if (is_dagger or is_repair) else "0",
    "ACT_EVAL_EPISODE_TIMESTAMP_OFFSETS": "49-80:2.0" if (is_dagger or is_repair) else "",
    "ACT_POLICY_PATH": str(MODEL_ROOT / recipe_model / "pretrained_model"),
}
print(json.dumps(protected_env, ensure_ascii=False, indent=2))
print("课程对照：repair15 15/30、stable61 7/30、protected DAgger 2/30。")


### 评估契约

`ACT_RECIPE=repair15` 使用 `chunk_size=20`、`n_action_steps=10`、timestamp/object-init state 和 `49-80:2.0` timestamp offset。评估模型路径指向当前配方的输出目录，逐回合结果写入 JSONL，并同步保存视频。


In [ ]:
# ACT_NATIVE_PROTECTED_TRAINING_CELL
# This is the protected ACT recipe itself. It runs in the Notebook kernel:
# dataset wrappers, weighted sampler, forward/backward, tqdm, checkpoints and metrics JSONL.
def train_act_repair15_in_notebook(repo_id, data_root, output_dir, init_policy_path, steps=2500, batch_size=16):
    import copy
    import io
    import json
    import time
    from collections import defaultdict
    from pathlib import Path

    import numpy as np
    import torch
    import torch.nn.functional as F
    import torchvision.transforms as T
    from PIL import Image
    from safetensors.torch import load_file
    from tqdm.auto import tqdm

    from lerobot.common.datasets.factory import resolve_delta_timestamps
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
    from lerobot.common.datasets.utils import dataset_to_policy_features
    from lerobot.configs.types import FeatureType, PolicyFeature
    from lerobot.policies.act.configuration_act import ACTConfig
    from lerobot.policies.act.modeling_act import ACTPolicy

    output_dir = Path(output_dir)
    data_root = Path(data_root)
    init_policy_path = Path(init_policy_path)
    output_dir.mkdir(parents=True, exist_ok=True)
    if not (init_policy_path / "model.safetensors").exists():
        raise FileNotFoundError(f"protected init weight missing: {init_policy_path}")

    episode_indexes = [0,1,2,3,4,5,6,8,9,10,13,14,15,16,17,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,40,41,42,43,44,45,47,48,50,51,52,53,54,55,56,57,58,59,60,61,62,63,65,67,68,69,71,72,73,74,75,76,77,78,79,80]
    timestamp_offsets = {episode: 2.0 for episode in range(49, 81)}

    def patch_image_transform():
        import lerobot.common.datasets.lerobot_dataset as ds_mod
        import lerobot.common.datasets.utils as ds_utils
        def hf_transform_to_torch(items_dict):
            to_tensor = T.ToTensor()
            for key in items_dict:
                first = items_dict[key][0]
                if isinstance(first, Image.Image):
                    items_dict[key] = [to_tensor(img) for img in items_dict[key]]
                elif isinstance(first, dict) and ("bytes" in first or "path" in first):
                    converted = []
                    for item in items_dict[key]:
                        if isinstance(item, dict) and item.get("bytes") is not None:
                            converted.append(to_tensor(Image.open(io.BytesIO(item["bytes"])).convert("RGB")))
                        else:
                            converted.append(item)
                    items_dict[key] = converted
                elif first is not None:
                    items_dict[key] = [item if isinstance(item, str) else torch.as_tensor(item) for item in items_dict[key]]
            return items_dict
        ds_utils.hf_transform_to_torch = hf_transform_to_torch
        ds_mod.hf_transform_to_torch = hf_transform_to_torch

    class EpisodeSubset(torch.utils.data.Dataset):
        def __init__(self, dataset, indexes):
            self.dataset = dataset
            self.indexes = []
            for episode in indexes:
                lo = int(dataset.episode_data_index["from"][episode].item())
                hi = int(dataset.episode_data_index["to"][episode].item())
                self.indexes.extend(range(lo, hi))
            if not self.indexes:
                raise ValueError("protected ACT episode subset is empty")
        def __len__(self):
            return len(self.indexes)
        def __getitem__(self, index):
            return self.dataset[self.indexes[int(index)]]

    class TimestampOffset(torch.utils.data.Dataset):
        def __init__(self, dataset, offsets):
            self.dataset, self.offsets = dataset, offsets
        def __len__(self):
            return len(self.dataset)
        def __getitem__(self, index):
            item = dict(self.dataset[int(index)])
            episode = item.get("episode_index", -1)
            if isinstance(episode, torch.Tensor):
                episode = int(episode.detach().cpu().reshape(-1)[0])
            if episode in self.offsets:
                timestamp = item["timestamp"]
                if isinstance(timestamp, torch.Tensor):
                    item["timestamp"] = timestamp + timestamp.new_tensor(self.offsets[episode])
                else:
                    item["timestamp"] = np.asarray(timestamp, dtype=np.float32) + np.float32(self.offsets[episode])
            return item

    class TimestampAndObjectInit(torch.utils.data.Dataset):
        def __init__(self, dataset):
            self.dataset = dataset
        def __len__(self):
            return len(self.dataset)
        def __getitem__(self, index):
            item = dict(self.dataset[int(index)])
            state = item["observation.state"]
            timestamp = torch.as_tensor(item["timestamp"], dtype=state.dtype).reshape(-1)
            obj_init = torch.as_tensor(item["obj_init"], dtype=state.dtype).reshape(-1)
            item["observation.state"] = torch.cat([state, timestamp, obj_init])
            return item

    def append_stats(stats, extra_key):
        patched = copy.deepcopy(stats)
        state = patched["observation.state"]
        extra = patched[extra_key]
        for key in ("min", "max", "mean", "std"):
            state[key] = np.concatenate([np.asarray(state[key]).reshape(-1), np.asarray(extra[key]).reshape(-1)])
        state["std"] = np.maximum(state["std"], 1e-6)
        return patched

    def stats_with_phase_and_object(stats, timestamp_values):
        patched = copy.deepcopy(stats)
        timestamp_values = np.asarray(timestamp_values, dtype=np.float32).reshape(-1)
        timestamp_stats = patched["timestamp"]
        for key in ("min", "max", "mean", "std"):
            if key == "min":
                value = timestamp_values.min()
            elif key == "max":
                value = timestamp_values.max()
            elif key == "mean":
                value = timestamp_values.mean()
            elif key == "std":
                value = max(float(timestamp_values.std()), 1e-6)
            patched["timestamp"][key] = np.asarray([value], dtype=np.float32)
        state = patched["observation.state"]
        for key in ("min", "max", "mean", "std"):
            state[key] = np.concatenate([np.asarray(state[key]).reshape(-1), np.asarray(patched["timestamp"][key]).reshape(-1)])
        return append_stats(patched, "obj_init")

    def episode_ids(dataset):
        values = []
        for index in range(len(dataset)):
            value = dataset[index].get("episode_index", -1)
            if isinstance(value, torch.Tensor):
                value = int(value.detach().cpu().reshape(-1)[0])
            values.append(int(value))
        return values

    def first_action(dataset, index):
        value = dataset[index]["action"]
        value = value.detach().cpu().numpy() if isinstance(value, torch.Tensor) else np.asarray(value)
        return value[0] if value.ndim == 2 else value

    def build_protected_sampler(dataset):
        ids = episode_ids(dataset)
        actions = np.stack([first_action(dataset, i) for i in range(len(dataset))]).astype(np.float64)
        weights = np.ones(len(dataset), dtype=np.float64)
        ranges = defaultdict(list)
        for index, episode in enumerate(ids):
            ranges[episode].append(index)
        for episode, indexes in ranges.items():
            if episode in timestamp_offsets:
                weights[indexes] *= 0.25
            weights[indexes[:250]] *= 6.0
        gripper = actions[:, 6]
        weights[gripper > 0.5] *= 6.0
        transitions = np.flatnonzero(np.abs(np.diff(gripper, prepend=gripper[0])) > 0.5)
        for transition in transitions:
            lo, hi = max(int(transition) - 24, 0), min(int(transition) + 25, len(weights))
            weights[lo:hi] *= 25.0
        motion = np.zeros(len(actions), dtype=np.float64)
        if len(actions) > 1:
            motion[1:] = np.linalg.norm(actions[1:, :6] - actions[:-1, :6], axis=1)
        scale = np.percentile(motion, 95) if np.any(motion > 0) else 0.0
        if scale > 1e-9:
            weights *= 1.0 + 8.0 * np.clip(motion / scale, 0.0, 1.0)
        sampler = torch.utils.data.WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True)
        info = {"mode":"trajectory", "frames":len(dataset), "episodes":len(ranges), "weight_min":float(weights.min()), "weight_max":float(weights.max()), "transition_count":int(len(transitions)), "offset_episodes":sorted(timestamp_offsets)}
        print("protected sampler =", json.dumps(info, ensure_ascii=False))
        return sampler, info

    patch_image_transform()
    metadata = LeRobotDatasetMetadata(repo_id, root=data_root)
    features = dataset_to_policy_features(metadata.features)
    output_features = {key: feature for key, feature in features.items() if feature.type is FeatureType.ACTION}
    input_features = {key: feature for key, feature in features.items() if feature.type is not FeatureType.ACTION}
    input_features.pop("observation.wrist_image", None)
    base_state = int(input_features["observation.state"].shape[0])
    input_features["observation.state"] = PolicyFeature(type=FeatureType.STATE, shape=(base_state + 7,))
    config = ACTConfig(
        input_features=input_features,
        output_features=output_features,
        chunk_size=20,
        n_action_steps=10,
        use_vae=False,
        dropout=0.0,
        kl_weight=10.0,
        optimizer_lr=1e-5,
        optimizer_weight_decay=1e-4,
        optimizer_lr_backbone=1e-5,
        device="cuda",
    )
    delta_timestamps = resolve_delta_timestamps(config, metadata)
    dataset = LeRobotDataset(repo_id, delta_timestamps=delta_timestamps, root=data_root)
    dataset = EpisodeSubset(dataset, episode_indexes)
    dataset = TimestampOffset(dataset, timestamp_offsets)
    timestamp_values = [float(torch.as_tensor(dataset[i]["timestamp"]).reshape(-1)[0]) for i in range(len(dataset))]
    stats = stats_with_phase_and_object(metadata.stats, timestamp_values)
    dataset = TimestampAndObjectInit(dataset)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    policy = ACTPolicy(config, dataset_stats=stats).to(device)
    state = load_file(str(init_policy_path / "model.safetensors"), device="cpu")
    missing, unexpected = policy.load_state_dict(state, strict=False)
    if missing or unexpected:
        raise RuntimeError(f"protected init mismatch: missing={missing}, unexpected={unexpected}")
    policy.train()
    optimizer = torch.optim.Adam(policy.parameters(), lr=1e-5)
    sampler, sampler_info = build_protected_sampler(dataset)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, sampler=sampler, num_workers=0, pin_memory=device.type == "cuda", drop_last=True)
    iterator = iter(loader)
    metrics_path = output_dir / "notebook_native_train_metrics.jsonl"
    manifest = {
        "recipe": "stable61_to_dagger_low_lr_step1500",
        "native_notebook_kernel": True,
        "repo_id": repo_id,
        "data_root": str(data_root),
        "init_policy_path": str(init_policy_path),
        "steps": int(steps),
        "batch_size": int(batch_size),
        "chunk_size": 20,
        "n_action_steps": 10,
        "use_vae": False,
        "optimizer": "Adam(lr=1e-5)",
        "state_features": "proprio + timestamp + obj_init",
        "sampler": sampler_info,
    }
    (output_dir / "notebook_native_train_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(manifest, ensure_ascii=False, indent=2))
    start = time.perf_counter()
    last = None
    for step in tqdm(range(1, int(steps) + 1), desc="ACT protected native Notebook train", dynamic_ncols=True):
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            batch = next(iterator)
        raw_action = batch["action"]
        batch = {key: (value.to(device, non_blocking=True) if isinstance(value, torch.Tensor) else value) for key, value in batch.items()}
        raw_action = raw_action.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        normalized = policy.normalize_inputs(batch)
        if policy.config.image_features:
            normalized = dict(normalized)
            normalized["observation.images"] = [normalized[key] for key in policy.config.image_features]
        normalized = policy.normalize_targets(normalized)
        actions_hat, _ = policy.model(normalized)
        valid = ~normalized["action_is_pad"]
        l1 = (F.l1_loss(normalized["action"], actions_hat, reduction="none") * valid.unsqueeze(-1)).mean()
        threshold = torch.tensor(0.5, dtype=torch.float32, device=device)
        action_buffer = policy.normalize_targets.buffer_action
        threshold_norm = (threshold - action_buffer["mean"][..., 6].to(device)) / (action_buffer["std"][..., 6].to(device) + 1e-8)
        target_bin = (raw_action[..., 6] > 0.5).to(dtype=actions_hat.dtype)
        logits = (actions_hat[..., 6] - threshold_norm) * 4.0
        bce_each = F.binary_cross_entropy_with_logits(logits, target_bin, reduction="none")
        valid_float = valid.to(dtype=bce_each.dtype)
        bce = (bce_each * valid_float).sum() / valid_float.sum().clamp_min(1.0)
        loss = l1 + 0.5 * bce
        loss.backward()
        optimizer.step()
        if step == 1 or step % 20 == 0 or step == int(steps):
            last = {"step":step, "loss":float(loss.detach().cpu()), "l1_loss":float(l1.detach().cpu()), "gripper_bce_loss":float(bce.detach().cpu()), "elapsed_s":float(time.perf_counter()-start)}
            with metrics_path.open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(last, ensure_ascii=False) + "\n")
            print(json.dumps(last, ensure_ascii=False))
        if step % 500 == 0 or step == int(steps):
            checkpoint = output_dir / f"step_{step}"
            policy.save_pretrained(checkpoint)
            print("saved native Notebook checkpoint:", checkpoint)
    policy.save_pretrained(output_dir)
    return {"output_dir": output_dir, "metrics_path": metrics_path, "manifest": output_dir / "notebook_native_train_manifest.json", "last": last}


In [5]:
# PROTECTED_TRAIN_CELL
protected_train_enabled = env_flag("RUN_PROTECTED_TRAIN", False)
if not protected_train_enabled:
    print("未启动。设置 RUN_PROTECTED_TRAIN=1 后，本单元会原生训练 ACT protected recipe；设置 ACT_RECIPE=repair15 可按当前保护协议准备 chunk20/n10 配置。")
else:
    DATASET_REPO_ID = globals().get("DATASET_REPO_ID", os.environ.get("ACT_DATASET_REPO_ID", "datawhale_eai_pnp_language"))
    TRAIN_DATA_ROOT = globals().get("TRAIN_DATA_ROOT", Path(os.environ.get("ACT_TRAIN_DATA_ROOT", DATA_ROOT / "demo_data_language")))
    CONFIG_DIR = OUTPUT_ROOT / "configs"
    RUN_ROOT = OUTPUT_ROOT / "runs" / "act_protected_current_recipe"
    is_repair = os.environ.get("ACT_RECIPE", "stable61").strip().lower() in {"repair15", "stable61_to_dagger", "nomemleak"}
    if is_repair:
        REPAIR_OUTPUT = RUN_ROOT / "repair15_continuation"
        initial_path = Path(os.environ.get("ACT_INITIAL_PROTECTED_PATH", str(MODEL_ROOT / "act_fallback_best_stable61_step2500")))
        repair_data_root = Path(os.environ.get("ACT_REPAIR_TRAIN_DATA_ROOT", TRAIN_DATA_ROOT))
        print("本单元在 Notebook kernel 内原生执行 repair15：不调用外部训练脚本。")
        native_result = train_act_repair15_in_notebook(
            repo_id=os.environ.get("ACT_DATASET_REPO_ID", DATASET_REPO_ID),
            data_root=repair_data_root,
            output_dir=REPAIR_OUTPUT,
            init_policy_path=initial_path,
            steps=int(os.environ.get("ACT_REPAIR_STEPS", "2500")),
            batch_size=int(os.environ.get("ACT_BATCH_SIZE", "16")),
        )
        ACT_REPAIR_NATIVE_OUTPUT = Path(native_result["output_dir"])
        ACT_REPAIR_NATIVE_METRICS = Path(native_result["metrics_path"])
        print("Notebook native training result:", public_path(ACT_REPAIR_NATIVE_OUTPUT))
        list_checkpoints(REPAIR_OUTPUT)
    else:
        STABLE_OUTPUT = RUN_ROOT / "stable61_full"
        stable_config = make_lerobot_train_config(
            "act",
            DATASET_REPO_ID,
            Path(os.environ.get("ACT_STABLE61_TRAIN_DATA_ROOT", TRAIN_DATA_ROOT)),
            STABLE_OUTPUT,
            steps=int(os.environ.get("ACT_STEPS", "5000")),
            batch_size=int(os.environ.get("ACT_BATCH_SIZE", "8")),
            chunk_size=50,
            n_action_steps=50,
        )
        stable_path = write_json_yaml(CONFIG_DIR / "act_protected_current_stable61.yaml", stable_config)
        train_lerobot_config_in_notebook(stable_path, enabled=True, progress_name="ACT protected-current stable61")
        print("protected-current candidate checkpoints:")
        list_checkpoints(STABLE_OUTPUT)


未启动。设置 RUN_PROTECTED_TRAIN=1 后，本单元会原生训练 ACT protected-current recipe。


In [ ]:
ACT_RECIPE = os.environ.get("ACT_RECIPE", "stable61").strip().lower()
ACT_IS_REPAIR = ACT_RECIPE in {"repair15", "stable61_to_dagger", "nomemleak"}
ACT_IS_DAGGER = ACT_RECIPE in {"dagger", "dagger_best025", "protected"} and not ACT_IS_REPAIR
ACT_DATASET_DEFAULT = "datawhale_eai_pnp_act_base72_plus_dagger3x3_rebuild_v1" if ACT_IS_REPAIR else ("datawhale_eai_pnp_act_dagger_best025_rebuild_v1" if ACT_IS_DAGGER else "datawhale_eai_pnp_language")
ACT_DATA_ROOT_DEFAULT = DATA_ROOT / ("act_base72_plus_dagger3x3_rebuild_v1" if ACT_IS_REPAIR else ("act_dagger_meta" if ACT_IS_DAGGER else "demo_data_language"))
ACT_MODEL_DEFAULT = "act_stable61_to_dagger_nomemleak_step1500_strict15of30" if ACT_IS_REPAIR else ("act_dagger_best025_rebuild_s5000" if ACT_IS_DAGGER else "act_stable61_step2500")
DATASET_REPO_ID = os.environ.get("ACT_DATASET_REPO_ID", ACT_DATASET_DEFAULT)
TRAIN_DATA_ROOT = Path(os.environ.get("ACT_TRAIN_DATA_ROOT", ACT_DATA_ROOT_DEFAULT))
ACT_POLICY_DEFAULT = MODEL_ROOT / ACT_MODEL_DEFAULT / "pretrained_model"
ACT_POLICY_PATH = Path(os.environ.get("ACT_POLICY_PATH", ACT_POLICY_DEFAULT))
CONFIG_DIR = OUTPUT_ROOT / "configs"
LOG_DIR = OUTPUT_ROOT / "logs"
RUN_ROOT = OUTPUT_ROOT / "runs" / "act_stable61_repro"
SMOKE_OUTPUT = RUN_ROOT / "smoke"
LONG_OUTPUT = RUN_ROOT / "stable61_full"

smoke_config = make_lerobot_train_config(
    "act", DATASET_REPO_ID, TRAIN_DATA_ROOT, SMOKE_OUTPUT,
    steps=2, batch_size=4, chunk_size=50, n_action_steps=50,
)
long_config = make_lerobot_train_config(
    "act", DATASET_REPO_ID, TRAIN_DATA_ROOT, LONG_OUTPUT,
    steps=int(os.environ.get("ACT_STEPS", "5000")),
    batch_size=int(os.environ.get("ACT_BATCH_SIZE", "8")),
    chunk_size=20 if ACT_IS_REPAIR else 50,
    n_action_steps=10 if ACT_IS_REPAIR else 50,
)
smoke_config_path = write_json_yaml(CONFIG_DIR / "act_smoke.yaml", smoke_config)
long_config_path = write_json_yaml(CONFIG_DIR / ("act_repair15_full.yaml" if ACT_IS_REPAIR else "act_stable61_full.yaml"), long_config)

train_lerobot_config_in_notebook(smoke_config_path, enabled=RUN_SMOKE, progress_name="ACT smoke")
train_lerobot_config_in_notebook(long_config_path, enabled=RUN_LONG_TRAIN, progress_name="ACT repair15 long train" if ACT_IS_REPAIR else "ACT long train")


## Checkpoint 3：实时查看训练日志和 checkpoint

            预计耗时：几秒。长训中可以重复运行本格，确认 step 是否推进、checkpoint 是否落盘。


In [7]:
print("smoke metrics:")
tail_log(SMOKE_OUTPUT / "notebook_train_metrics.jsonl", lines=20)
print("\nlong train metrics:")
tail_log(LONG_OUTPUT / "notebook_train_metrics.jsonl", lines=40)
print("\ncheckpoints:")
list_checkpoints(LONG_OUTPUT)


smoke metrics:
日志不存在： $OUTPUT_ROOT/runs/act_stable61_repro/smoke/notebook_train_metrics.jsonl

long train metrics:
日志不存在： $OUTPUT_ROOT/runs/act_stable61_repro/stable61_full/notebook_train_metrics.jsonl

checkpoints:
尚未发现 checkpoint： $OUTPUT_ROOT/runs/act_stable61_repro/stable61_full


## Checkpoint 4：课程权重对照

下表汇总课程中已经完成的 ACT 训练阶段。当前运行产生的新 checkpoint 继续使用后续严格评估单元选择。


In [8]:
rows = [
    ("stage1 stable42", "8000 steps", "可训练但闭环仍不稳"),
    ("stable61 fallback", "5000 steps", "step2500 strict30 为 7/30"),
    ("repair15 continuation", "2500 steps", "step1500 strict30 为 15/30；3/10 + 4/10 + 8/10"),
]
md_table(["阶段", "训练进度", "结论"], rows)


| 阶段 | 训练进度 | 结论 |
| --- | --- | --- |
| stage1 stable42 | 8000 steps | 可训练但闭环仍不稳 |
| stable61 | 5000 steps | step2500 优于 step5000 |
| strict30 | 15/30 | 当前保护候选；三面板 3/10 + 4/10 + 8/10 |

## Checkpoint 5：Notebook 内严格评估

            预计耗时：30 个 episode 往往需要较久；可以先用 `ACT_EVAL_EPISODES=6` 做快速门禁，再扩到 30。  
            本单元会在 Notebook kernel 内直接加载 ACT 并逐 seed rollout；不再 shell 到外部训练/评估脚本。


In [ ]:
act_recipe = os.environ.get("ACT_RECIPE", "stable61").strip().lower()
act_is_repair = act_recipe in {"repair15", "stable61_to_dagger", "nomemleak"}
act_is_dagger = act_recipe in {"dagger", "dagger_best025", "protected"} and not act_is_repair
if act_is_repair or act_is_dagger:
    os.environ.setdefault("ACT_EVAL_N_ACTION_STEPS", "10")
    os.environ.setdefault("ACT_EVAL_CLAMP_TIMESTAMP", "1")
    os.environ.setdefault("ACT_EVAL_EPISODE_TIMESTAMP_OFFSETS", "49-80:2.0")
eval_episodes = os.environ.get("ACT_EVAL_EPISODES", "30")
eval_seed_start = os.environ.get("ACT_EVAL_SEED_START", "1030")
result_stem = "act_repair15" if act_is_repair else ("act_dagger_best025" if act_is_dagger else "act_stable61")
result_path = OUTPUT_ROOT / f"{result_stem}_seed{eval_seed_start}_{int(eval_episodes)}ep.jsonl"
if RUN_EVAL:
    default_policy = Path(os.environ.get("ACT_EVAL_POLICY_PATH", str(globals().get("ACT_REPAIR_NATIVE_OUTPUT", ACT_POLICY_PATH))))
    eval_policy = resolve_eval_policy(default_policy, LONG_OUTPUT, "ACT_EVAL_POLICY_PATH")
    eval_repo_id = os.environ.get("ACT_EVAL_REPO_ID", DATASET_REPO_ID)
    eval_data_root = Path(os.environ.get("ACT_EVAL_DATA_ROOT", str(TRAIN_DATA_ROOT)))
    run_eval_policy_in_notebook(
        "act",
        eval_policy,
        result_path,
        episodes=eval_episodes,
        seed_start=eval_seed_start,
        render=env_flag("RENDER_EVAL"),
        enabled=True,
        repo_id=eval_repo_id,
        dataset_root=eval_data_root,
    )
    summarize_jsonl(result_path)
else:
    print("RUN_EVAL=0: skipped closed-loop evaluation.")
    print("Set ACT_EVAL_POLICY_PATH and RUN_EVAL=1 to evaluate the model produced by this workflow.")


## Checkpoint 6：阶段表现与改进方向

In [10]:
rows = [
    ("接近/接触", "经常能做到", "已形成可用的视觉—动作关联"),
    ("搬运/对准/释放", "最容易失败", "长时序行为克隆会累积误差"),
    ("loss", "用于观察优化过程", "与 strict physical_success 和视频联合选择 checkpoint"),
    ("继续方向", "更高质量 success-only / recovery 数据", "优先补充失败阶段的状态覆盖"),
]
md_table(["观察", "现象", "处理方式"], rows)
show_image("act_success_sequence.jpg", "ACT 成功关键帧")
show_image("act_failure_sequence.jpg", "ACT 失败关键帧")


| 观察 | 现象 | 处理方式 |
| --- | --- | --- |
| 接近/接触 | 经常能做到 | 说明不是完全没有视觉或动作能力 |
| 搬运/对准/释放 | 最容易失败 | 长时序行为克隆会累积误差 |
| loss | 不能单独决定 checkpoint | 必须用 strict physical_success + 视频 |
| 继续方向 | 更高质量 success-only / recovery 数据 | 不建议盲目扩大模型参数 |

**ACT 成功关键帧**

**ACT 失败关键帧**

## Checkpoint 7：写入教程时的结论


In [ ]:
print("""
ACT 课程结果：
- repair15：三个固定面板合计 15/30（3/10 + 4/10 + 8/10）。
- stable61 参考结果：7/30。
- protected DAgger 对照：2/30。

设置 ACT_RECIPE=repair15 可载入对应训练配置。评估单元读取当前工作流生成的策略，并保存逐回合结果和视频。
""")
